In [1]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

# Import libraries/modules to use
import numpy as np
import pandas as pd
import joblib as jb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, median_absolute_error, root_mean_squared_error
from sklearn.linear_model import Ridge

Mounted at /content/drive


In [2]:
# Load in the data
filepath = '/content/drive/MyDrive/ACP_Project_28A/PivotedData.csv'
df_pivoted = pd.read_csv(filepath)

# Convert date to datetime before trying to resort
df_pivoted['date'] = pd.to_datetime(df_pivoted['date'])

# Enforce that the pivoted dataframe must be ordered chronologically and drop the residual old index
df_pivoted = df_pivoted.sort_values(['date', 'hour']).reset_index(drop=True)

# Add a Boolean is_weekend column
df_pivoted['is_weekend'] = df_pivoted['day_of_week'].isin([6, 7]).astype(int)

# Convert station_key to categorical before encoding
df_pivoted['station_key'] = df_pivoted['station_key'].astype(str)

# Check that all missing values are filled, number of unique values are correct and datatypes are correct
display(df_pivoted.isna().sum())
display(df_pivoted.nunique())
display(df_pivoted.dtypes)

display(df_pivoted)

,0
date,0
station_key,0
traffic_direction_seq,0
cardinal_direction_seq,0
month,0
day_of_week,0
school_holiday,0
hour,0
traffic_vol,0
avg_prev3h,0


,0
date,731
station_key,101
traffic_direction_seq,2
cardinal_direction_seq,4
month,12
day_of_week,7
school_holiday,2
hour,24
traffic_vol,4335
avg_prev3h,9878


,0
date,"datetime64[ns, UTC]"
station_key,object
traffic_direction_seq,int64
cardinal_direction_seq,int64
month,int64
day_of_week,int64
school_holiday,int64
hour,int64
traffic_vol,int64
avg_prev3h,float64


,date,station_key,traffic_direction_seq,cardinal_direction_seq,month,day_of_week,school_holiday,hour,traffic_vol,avg_prev3h,avg_prevday,is_weekend
0,2023-01-01 00:00:00+00:00,55385,1,3,1,7,0,0,23,23.000,1334.417,1
1,2023-01-01 00:00:00+00:00,55393,0,3,1,7,0,0,811,811.000,592.542,1
2,2023-01-01 00:00:00+00:00,55430,1,1,1,7,0,0,202,202.000,364.938,1
3,2023-01-01 00:00:00+00:00,55430,0,5,1,7,0,0,378,378.000,364.938,1
4,2023-01-01 00:00:00+00:00,55432,0,3,1,7,0,0,353,353.000,381.667,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2824411,2024-12-31 00:00:00+00:00,99990004,0,3,12,2,0,23,207,101.333,166.396,0
2824412,2024-12-31 00:00:00+00:00,99990004,1,7,12,2,0,23,20,165.000,166.396,0
2824413,2024-12-31 00:00:00+00:00,99990004,1,7,12,2,0,23,245,81.333,166.396,0
2824414,2024-12-31 00:00:00+00:00,99990010,0,3,12,2,0,23,945,393.000,443.875,0


In [6]:
# Extract the feature set (X) and the target (y)
X = df_pivoted[['station_key', 'traffic_direction_seq', 'cardinal_direction_seq', 'month', 'day_of_week', 'school_holiday', 'hour', 'avg_prev3h', 'avg_prevday', 'is_weekend']]
y = df_pivoted['traffic_vol']

# One-hot encoder to encode the station_key into a numerical matrix at training, testing and actual use time - feed into Pipeline
encoder = ColumnTransformer(transformers=[('station_encode', OneHotEncoder(handle_unknown='ignore'), ['station_key'])], remainder='passthrough')

# Adjustable parameter values for the Regressor
a = 1000
max_iters = 1000

# Set up Pipeline with the encoding operation and the model
ridgemodel = Pipeline([
    ("encode", encoder),
    ("train", Ridge(random_state=17, alpha=a, tol=1e-6, max_iter=max_iters))
])

In [7]:
# Create TimeSeriesSplit for cross-validation
tss = TimeSeriesSplit(n_splits=5)

# Keep track of the mean absolute error with each iteration
mae_track = []
medae_track = []
rmse_track = []

In [8]:
%%time
# For each split from the TimeSeriesSplit, extract train and test sets for features and target, train an interim model, check its predicted values against the actual values and append the mean absolute error between the actual and predicted values to the track
print(f'Testing for alpha={a}, max_iter={max_iters}: ')
for train_sub, test_sub in tss.split(X):
    X_train, X_test = X.iloc[train_sub], X.iloc[test_sub]
    y_train, y_test = y.iloc[train_sub], y.iloc[test_sub]
    # Log-transform training targets
    y_train_log = np.log1p(y_train)
    print('Log transform applied to targets!')
    # Train a test/sample model with current parameters
    ridgemodel.fit(X_train, y_train_log)
    print('Test model trained!')
    # Get predictions from the now trained model to test
    predictions_log = ridgemodel.predict(X_test)
    predictions = np.expm1(predictions_log)
    print('Exp transform applied to predictions!')
    predictions = ridgemodel.predict(X_test)
    # Tack on the mean absolute error onto the track
    mae_track.append(mean_absolute_error(y_test, predictions))
    medae_track.append(median_absolute_error(y_test, predictions))
    rmse_track.append(root_mean_squared_error(y_test, predictions))
    print('Scores added to tracks!')

# Show the running track for evaluation
metrics = pd.DataFrame(columns=['0', '1', '2', '3', '4'], index=['mae', 'medae', 'rmse'])
metrics.loc['mae'] = mae_track
metrics.loc['medae'] = medae_track
metrics.loc['rmse'] = rmse_track
avg_metrics = metrics.mean(axis=1)
metrics['Avg'] = avg_metrics

display(metrics)

outpath = f'/content/drive/MyDrive/ACP_Project_28A/LRMetrics/RidgeMetrics_tol6_{a}_{max_iters}.csv'
metrics.to_csv(outpath)

Testing for alpha=1000, max_iter=1000: 
Log transform applied to targets!
Test model trained!
Exp transform applied to predictions!
Scores added to tracks!
Log transform applied to targets!
Test model trained!
Exp transform applied to predictions!
Scores added to tracks!
Log transform applied to targets!
Test model trained!
Exp transform applied to predictions!
Scores added to tracks!
Log transform applied to targets!
Test model trained!
Exp transform applied to predictions!
Scores added to tracks!
Log transform applied to targets!
Test model trained!
Exp transform applied to predictions!
Scores added to tracks!


,0,1,2,3,4,Avg
mae,361.876562,373.449766,331.048545,327.459281,322.830019,343.332835
medae,85.03785,90.76377,80.248902,77.823464,73.799341,81.534665
rmse,674.126083,686.876451,621.304356,611.840769,612.671965,641.363925


CPU times: user 27.7 s, sys: 1.23 s, total: 28.9 s
Wall time: 22.3 s


In [ ]:
# With parameters figured out, prepare to train the final production model - log-transform before training; remember to exp-transform prediction
# Before running this, check that:
# 1. You have your final parameters figured out and entered above
# 2. That you ran the cell with the final parameters just before coming down to this one - do not run testing loop
def train_model(model, X, y):
  y_log = np.log1p(y)
  model.fit(X, y_log)
  return model

In [ ]:
# Train the model
export = train_model(ridgemodel, X, y)

In [ ]:
# Package and export the trained model
jb.dump(export, 'ridgemodel.joblib', compress=('gzip', 3))